In [ ]:
import numpy as np
from scipy.io.wavfile import read
from scipy.signal import stft

INPUT = "soln.wav"
CHARSET = "abcdefghijklmnopqrstuvwxyz0123456789{}"

SLOT_TIME = 0.6
BEEP_TIME = 0.4

def get_fibonacci(n):
    fib = [1, 1]
    for _ in range(n):
        fib.append(fib[-1] + fib[-2])
    return fib

sample_rate, audio = read(INPUT)

audio = audio.astype(float) / np.max(np.abs(audio))
if audio.ndim > 1:
    audio = audio.mean(axis=1)

frequencies, times, Zxx = stft(audio, fs=sample_rate, nperseg=4096, noverlap=2048)
magnitude = np.abs(Zxx)

freq_mask = frequencies >= 14500
filtered_freqs = frequencies[freq_mask]
filtered_mag = magnitude[freq_mask, :]

THRESHOLD = np.max(filtered_mag) * 0.5

num_slots = int(times[-1] / SLOT_TIME)
detected_frequencies = []
recording = False

print("Scanning for the 15,000 Hz Anchor Note...\n")

for i in range(num_slots):
    start_t = i * SLOT_TIME
    end_t = start_t + BEEP_TIME
    
    time_mask = (times >= start_t) & (times <= end_t)
    chunk = filtered_mag[:, time_mask]
    
    if chunk.size > 0:
        peak_volume = np.max(chunk)
        max_idx = np.unravel_index(np.argmax(chunk), chunk.shape)
        
        if peak_volume > THRESHOLD:
            raw_peak_freq = filtered_freqs[max_idx[0]]
            clean_freq = round(raw_peak_freq / 25) * 25
            
            if not recording:
                if clean_freq == 15000:
                    print(f"Anchor found at {start_t:.1f}s! Tracking sequence...")
                    recording = True
                else:
                    continue
            
            if recording:
                detected_frequencies.append(clean_freq)
        else:
            if recording:
                print(f"Sequence ended at {start_t:.1f}s. Halting scan.")
                break

if not detected_frequencies:
    print("No frequencies found.")
else:
    fib_key = get_fibonacci(len(detected_frequencies))
    
    raw_numbers = []
    decoded_flag = ""
    
    print("-" * 40)
    print("Decrypting intervals...\n")
    
    for i in range(1, len(detected_frequencies)):
        current = detected_frequencies[i]
        previous = detected_frequencies[i-1]
        
        jump = abs(current - previous)
        encrypted_val = int(jump / 25)
        
        fib_val = fib_key[i - 1]
        char_index = encrypted_val ^ (fib_val % 128)
        
        raw_numbers.append(char_index)
        
        if 1 <= char_index <= len(CHARSET):
            decoded_flag += CHARSET[char_index - 1]
        else:
            decoded_flag += "?"
            
    print(f"1. Raw Decrypted Indices: {raw_numbers}")
    print(f"2. Mapped to Alphabet:    {decoded_flag}")
    print("-" * 40)